# 03 — Building Permits: Preparation and Analysis

**Phase:** Building Permits Analysis  
**Notebook goal:** Understand the structure and category landscape of the permits dataset and establish a defensible residential-filtering rule before any aggregation is performed.  
**Status:** Exploration only — no filtering, no aggregations yet.

---

## 1. Purpose

This notebook begins the **Building Permits Analysis** block of the project.

The goal is to understand the category landscape of the Issued Building Permits dataset well enough to write a defensible residential-filtering rule. That rule will be applied in a later step before permit counts and project values are aggregated by year.

Specifically, this notebook:

1. Loads a 1,000-row sample of the raw permits file and confirms the required columns are present.
2. Reports the date range covered by the sample.
3. Explores the category columns — `TypeOfWork`, `PermitCategory`, `PropertyUse`, and `SpecificUseCategory` — to understand how permits are classified.
4. Identifies the decision that must be made before aggregation can proceed.

## 2. Analytical Grain

**One row = one permit record.**

Each row in the Issued Building Permits dataset represents a single building permit that was issued by the City of Vancouver. One property can have multiple permits issued at different times (for example, a demolition permit followed later by a new-construction permit).

**Permit issuance is a supply signal, not a completed housing unit.**

A permit is issued at the start of a regulated construction or renovation activity. It does not confirm that work was completed, that a unit was occupied, or that a dwelling was added to the housing stock. Permits are used here as a leading indicator of housing supply activity — they signal intent and approval, not delivery.

**The residential filtering rule must be based on observed categories, not assumptions.**

The dataset contains permits for residential, commercial, industrial, and institutional work. Before aggregating permit counts or project values as a housing supply signal, we must decide which rows to include. That decision must be grounded in the actual category values present in the data — not assumed in advance. This notebook exists to gather the evidence needed to make that decision.

## 3. Input

| Item | Detail |
|---|---|
| Source file | `data/raw/issued_building_permits_raw.csv` |
| Rows loaded | 1,000 (sample only) |
| Delimiter | `;` (semicolon) |
| Encoding | `utf-8-sig` (strips byte-order mark) |

Required columns for this analysis:

| Column | Role |
|---|---|
| `IssueDate` | Full date the permit was issued |
| `IssueYear` | Year extracted from `IssueDate` |
| `YearMonth` | Year-month string for finer time grouping |
| `ProjectValue` | Declared value of the permitted work |
| `TypeOfWork` | Broad work category (e.g. new construction, alteration) |
| `PermitCategory` | More specific permit classification |
| `PropertyUse` | High-level land use (e.g. Dwelling Uses) |
| `SpecificUseCategory` | Detailed use within the property use group |
| `GeoLocalArea` | City neighbourhood for geographic grouping |

### Import Libraries and Define Constants

We import `pandas` and `os`. Column names needed later in the notebook are stored as a list constant so they can be referenced consistently without retyping. The file path is built from the project root directory rather than hard-coded, so the notebook runs correctly regardless of where the repository is cloned.

In [1]:
import os
import pandas as pd

BASE_DIR     = os.path.abspath(os.path.join(os.getcwd(), '..'))
PERMITS_PATH = os.path.join(BASE_DIR, 'data', 'raw', 'issued_building_permits_raw.csv')

REQUIRED_COLS = [
    'IssueDate',
    'IssueYear',
    'YearMonth',
    'ProjectValue',
    'TypeOfWork',
    'PermitCategory',
    'PropertyUse',
    'SpecificUseCategory',
    'GeoLocalArea',
]

SAMPLE_ROWS = 1_000

print('Setup complete.')
print(f'File path : {PERMITS_PATH}')

Setup complete.
File path : c:\Users\User\Documents\vancouver-property-value-housing-supply-analysis\data\raw\issued_building_permits_raw.csv


### Load a 1,000-Row Sample

We raise an explicit error if the file is missing before attempting to load, so the failure message is informative rather than a generic `FileNotFoundError` from pandas. After loading, we print the shape to confirm both the row count and column count are as expected.

In [2]:
if not os.path.isfile(PERMITS_PATH):
    raise FileNotFoundError(
        f'Raw permits file not found. Expected: {PERMITS_PATH}'
    )

df = pd.read_csv(
    PERMITS_PATH,
    sep=';',
    nrows=SAMPLE_ROWS,
    encoding='utf-8-sig',
    low_memory=False,
)

print(f'File found  : {PERMITS_PATH}')
print(f'Sample shape: {df.shape}  →  ({df.shape[0]:,} rows, {df.shape[1]} columns)')

File found  : c:\Users\User\Documents\vancouver-property-value-housing-supply-analysis\data\raw\issued_building_permits_raw.csv
Sample shape: (1000, 20)  →  (1,000 rows, 20 columns)


## 4. Initial Validation

Before exploring any distributions, we confirm that all required columns are present and report the missing-value rate for each. A required column that is absent or heavily null would change the analysis approach before any further work is done.

### Required Column Check

We loop over the list of required columns and check whether each one appears in the DataFrame. A `MISSING` status here means the column was not in the file at all — possibly renamed or not included in this export. A `FOUND` status means the column loaded correctly, though it may still contain null values (checked next).

In [3]:
all_present = True
for col in REQUIRED_COLS:
    status = 'FOUND  ' if col in df.columns else 'MISSING'
    if status == 'MISSING':
        all_present = False
    print(f'  {status}: {col}')

print()
print(f'All required columns present: {all_present}')

  FOUND  : IssueDate
  FOUND  : IssueYear
  FOUND  : YearMonth
  FOUND  : ProjectValue
  FOUND  : TypeOfWork
  FOUND  : PermitCategory
  FOUND  : PropertyUse
  FOUND  : SpecificUseCategory
  FOUND  : GeoLocalArea

All required columns present: True


### Missing-Value Counts for Required Columns

We count null values in each required column across the 1,000-row sample. Columns with high null rates may need special handling when the residential filter or aggregations are defined. `PermitCategory` in particular is worth watching — if it is heavily null, it may not be reliable as the sole basis for a residential filter.

In [4]:
missing_counts = df[REQUIRED_COLS].isnull().sum()
missing_pct    = (missing_counts / len(df) * 100).round(1)

missing_summary = pd.DataFrame({
    'missing_count': missing_counts,
    'missing_pct'  : missing_pct,
}).sort_values('missing_count', ascending=False)

display(missing_summary)

,missing_count,missing_pct
PermitCategory,429,42.9
GeoLocalArea,4,0.4
IssueDate,0,0.0
IssueYear,0,0.0
YearMonth,0,0.0
TypeOfWork,0,0.0
ProjectValue,0,0.0
PropertyUse,0,0.0
SpecificUseCategory,0,0.0


## 5. Date Coverage

We report the minimum and maximum `IssueYear` present in the sample. This tells us the approximate time span represented, which is important for deciding how many years of permit data will be available when the analysis is extended to the full dataset.

Because we are working with a 1,000-row sample, early or late years may be under-represented. The full date range should be verified when the full dataset is loaded.

In [5]:
min_year = df['IssueYear'].min()
max_year = df['IssueYear'].max()
unique_years = sorted(df['IssueYear'].dropna().unique())

print(f'Minimum IssueYear in sample : {min_year}')
print(f'Maximum IssueYear in sample : {max_year}')
print(f'Unique years in sample      : {unique_years}')

Minimum IssueYear in sample : 2017
Maximum IssueYear in sample : 2026
Unique years in sample      : [np.int64(2017), np.int64(2018), np.int64(2019), np.int64(2025), np.int64(2026)]


In [6]:
# Load only the IssueYear column from the complete file.
# This is memory-efficient because we are not loading the other 19 columns.
full_years = pd.read_csv(PERMITS_PATH,sep=';',usecols=['IssueYear'],encoding='utf-8-sig',low_memory=False)

year_counts = (full_years['IssueYear'].value_counts(dropna=False).sort_index())

print("Minimum IssueYear:", full_years['IssueYear'].min())
print("Maximum IssueYear:", full_years['IssueYear'].max())
print("\nPermit records by year:")
display(year_counts)

Minimum IssueYear: 2017
Maximum IssueYear: 2026

Permit records by year:


IssueYear
2017    6718
2018    6750
2019    5564
2020    4389
2021    5020
2022    5942
2023    4684
2024    4915
2025    5145
2026    1868
Name: count, dtype: int64

## 6. Residential Classification Review

The following four columns are candidates for defining a residential filter. We inspect each one to understand the vocabulary used in the dataset. The output of this section is the evidence base for the decision in the next section.

For columns with a small number of unique values (`TypeOfWork`, `PermitCategory`), we display every value and its count. For columns with many unique values (`PropertyUse`, `SpecificUseCategory`), we display the top 20 by frequency. In all cases, null values are included in the count so we can see the full picture.

### TypeOfWork

`TypeOfWork` describes the broad category of construction activity being permitted. Common values across building permit datasets include new construction, additions, alterations, demolitions, and salvage or abatement work. Understanding which values appear here helps us decide whether this column alone is sufficient to identify residential supply, or whether it must be combined with a use-based column.

`value_counts(dropna=False)` is used throughout this section so null values appear in the table rather than being silently excluded.

In [7]:
type_of_work_counts = df['TypeOfWork'].value_counts(dropna=False)
print(f'Unique values in TypeOfWork (including null): {df["TypeOfWork"].nunique(dropna=False)}')
print()
display(type_of_work_counts.to_frame('count'))

Unique values in TypeOfWork (including null): 6



,count
TypeOfWork,
Addition / Alteration,456
New Building,241
Demolition / Deconstruction,167
Salvage and Abatement,124
Temporary Building / Structure,10
Outdoor Uses (No Buildings Proposed),2


In [8]:
# Load only TypeOfWork from the complete file.

full_type_of_work = pd.read_csv(PERMITS_PATH,sep=';',usecols=['TypeOfWork'],encoding='utf-8-sig',low_memory=False)

type_of_work_counts = (full_type_of_work['TypeOfWork'].value_counts(dropna=False))

print("TypeOfWork categories in the complete dataset:")
display(type_of_work_counts)

TypeOfWork categories in the complete dataset:


TypeOfWork
Addition / Alteration                   25068
New Building                            11684
Demolition / Deconstruction              6948
Salvage and Abatement                    6749
Temporary Building / Structure            484
Outdoor Uses (No Buildings Proposed)       62
Name: count, dtype: int64

### PermitCategory

`PermitCategory` is a more specific classification than `TypeOfWork`. It may distinguish residential from commercial work within the same broad category, making it a strong candidate for a residential filter. However, if it has a high null rate (as suggested by the missing-value check above), any filter based solely on this column will produce a result with a significant gap where the category is unknown.

In [10]:
permit_category_counts = df['PermitCategory'].value_counts(dropna=False)
print(f'Unique values in PermitCategory (including null): {df["PermitCategory"].nunique(dropna=False)}')
print()
display(permit_category_counts.to_frame('count'))

Unique values in PermitCategory (including null): 5



,count
PermitCategory,
NaN,429
New Build - Low Density Housing,179
Renovation - Residential - Lower Complexity,178
Renovation - Commercial/ Mixed Use - Lower Complexity,170
New Build - Standalone Laneway,44


### PropertyUse

`PropertyUse` describes the intended use of the property to which the permit applies. A value of `Dwelling Uses` (or similar) is the clearest direct signal that a permit relates to residential housing. This column may be more consistently populated than `PermitCategory` and could serve as a reliable primary filter or as a complement to it.

We display the top 20 values by frequency, including nulls.

In [11]:
property_use_counts = df['PropertyUse'].value_counts(dropna=False)
print(f'Unique values in PropertyUse (including null): {df["PropertyUse"].nunique(dropna=False)}')
print()
display(property_use_counts.head(20).to_frame('count'))

Unique values in PropertyUse (including null): 30



,count
PropertyUse,
Dwelling Uses,733
Office Uses,110
Retail Uses,42
Service Uses,36
Institutional Uses,19
Cultural/Recreational Uses,15
Manufacturing Uses,9
Wholesale Uses,5
Transportation and Storage Uses,5


In [13]:
# Load only PropertyUse from the complete dataset.


full_property_use = pd.read_csv(PERMITS_PATH,sep=';',usecols=['PropertyUse'],encoding='utf-8-sig',low_memory=False)

property_use_counts_full = (full_property_use['PropertyUse'].value_counts(dropna=False).head(30))

print("Top 30 PropertyUse values in the complete dataset:")
display(property_use_counts_full.to_frame('count'))

Top 30 PropertyUse values in the complete dataset:


,count
PropertyUse,
Dwelling Uses,35582
Office Uses,6419
Retail Uses,2455
Service Uses,2093
Institutional Uses,1031
Cultural/Recreational Uses,900
Manufacturing Uses,366
Wholesale Uses,206
"Office Uses,Retail Uses",183


### SpecificUseCategory

`SpecificUseCategory` provides the most granular classification of the permitted use. Within `Dwelling Uses`, it may distinguish between single-detached houses, apartments, townhouses, and other dwelling types. This level of detail may be useful for disaggregating housing supply by type in a later analysis phase, or for excluding non-residential dwelling categories (such as care facilities or hotels).

We display the top 20 values by frequency, including nulls.

In [12]:
specific_use_counts = df['SpecificUseCategory'].value_counts(dropna=False)
print(f'Unique values in SpecificUseCategory (including null): {df["SpecificUseCategory"].nunique(dropna=False)}')
print()
display(specific_use_counts.head(20).to_frame('count'))

Unique values in SpecificUseCategory (including null): 84



,count
SpecificUseCategory,
Single Detached House,323
Single Detached House w/Sec Suite,127
Multiple Dwelling,126
Laneway House,94
General Office,89
Duplex,32
Retail Store,30
Restaurant - Class 1,19
Health Care Office,16


In [14]:
# Load only SpecificUseCategory from the complete dataset.
# This helps identify the concrete residential uses represented.

full_specific_use = pd.read_csv( PERMITS_PATH,sep=';',usecols=['SpecificUseCategory'],encoding='utf-8-sig',low_memory=False)

specific_use_counts_full = (full_specific_use['SpecificUseCategory'].value_counts(dropna=False).head(40))

print("Top 40 SpecificUseCategory values in the complete dataset:")
display(specific_use_counts_full.to_frame('count'))

Top 40 SpecificUseCategory values in the complete dataset:


,count
SpecificUseCategory,
Single Detached House,13800
Multiple Dwelling,6588
Single Detached House w/Sec Suite,6001
General Office,5373
Laneway House,4180
Duplex,2112
Retail Store,1986
Restaurant - Class 1,947
Dwelling Unit,805


In [15]:
# Load the two columns needed to validate the residential filter.
full_residential_review = pd.read_csv(PERMITS_PATH,sep=';',usecols=['PropertyUse', 'SpecificUseCategory'],encoding='utf-8-sig',low_memory=False)

# Candidate rule: include any permit whose PropertyUse contains Dwelling Uses.
residential_mask = (full_residential_review['PropertyUse'].astype('string').str.contains('Dwelling Uses', case=False, na=False))

residential_count = residential_mask.sum()
total_count = len(full_residential_review)
residential_pct = residential_count / total_count * 100

print(f"Total permit records        : {total_count:,}")
print(f"Residential permit records  : {residential_count:,}")
print(f"Residential share           : {residential_pct:.1f}%")

print("\nTop residential SpecificUseCategory values:")
display(full_residential_review.loc[residential_mask, 'SpecificUseCategory'].value_counts(dropna=False).head(30).to_frame('count'))

Total permit records        : 50,995
Residential permit records  : 36,201
Residential share           : 71.0%

Top residential SpecificUseCategory values:


,count
SpecificUseCategory,
Single Detached House,13800
Multiple Dwelling,6588
Single Detached House w/Sec Suite,6001
Laneway House,4180
Duplex,2112
Dwelling Unit,805
Duplex w/Secondary Suite,715
Multiple Conversion Dwelling,385
Infill Single Detached House,210


## 7. Decision Required

Before aggregation can proceed, a residential-filtering rule must be defined and documented. The exploration above provides the evidence base. The decision has the following dimensions:

**Which column(s) define the residential filter?**

| Column | Strength | Risk |
|---|---|---|
| `PropertyUse` | Direct use signal; likely well-populated | May include non-housing dwelling uses |
| `PermitCategory` | Specific and descriptive | Significant null rate — gaps in coverage |
| `TypeOfWork` | Available but broad | Does not distinguish residential from commercial |
| `SpecificUseCategory` | Most granular | Many values; requires an explicit inclusion list |

**Which values within the chosen column(s) count as residential?**

The full-dataset value counts (not visible in this 1,000-row sample) are needed to confirm the complete vocabulary. The values observed here are indicative only.

**Should the filter target new construction only, or all permit types?**

For a housing supply signal, new-construction permits are the most direct measure. Addition and alteration permits add density to existing stock in a different way. The analysis scope should be clarified before the filter is set.

**Action required:** Document the chosen filter in `docs/dataset_decision_log.md` before implementing it in this notebook. The decision log entry should state the chosen column(s), the included values, the rationale, and any known limitations.

## 8. Next Step

Once the residential filter decision is documented in `docs/dataset_decision_log.md`, the following steps will be added to this notebook:

1. **Load the full dataset** — reload without `nrows` to work with all permit records.
2. **Apply the residential filter** — retain only rows matching the agreed category values.
3. **Validate the filter** — confirm the row count after filtering and review a sample of filtered-out rows to check for false positives and false negatives.
4. **Derive `permit_count_by_year`** — count residential permits issued per year.
5. **Derive `permit_project_value_by_year`** — sum `ProjectValue` for residential permits per year as an alternative supply-intensity measure.

Both aggregations will be saved to `data/processed/` for use in the cross-dataset analysis phase.

---

## `permit_count_by_year` — Annual Residential Permit Count

With the residential filter decision documented, we now load the full permits dataset, apply the agreed filter, and produce the first annual supply metric. The sections that follow build the metric step by step and validate it before use.

## 9. Business Definition — `permit_count_by_year`

**`permit_count_by_year`** is the annual count of unique residential building permits issued in the City of Vancouver, covering the years 2019 through 2024.

Three important framing points:

- **Permit count represents issued permit identifiers, not completed housing units.** A permit is issued at the start of a regulated construction activity. It does not confirm that work was started, finished, or that a dwelling was made available.

- **A single permit may cover one or many dwelling units.** A new-apartment-building permit and a single-detached-house permit both count as one permit in this metric. The metric captures the breadth of permitting activity, not the volume of units.

- **This is a housing-supply activity signal, not a direct count of new homes.** It indicates the pace at which the city approved residential construction work, which is useful as a leading indicator of supply pressure.

## 10. Analytical Grain

| Level | Grain | Description |
|---|---|---|
| Source data | One row = one permit record | Each row in the raw file is one permit |
| Output metric | One row = one calendar year | Aggregated to year level |

The grain shifts from permit-level to year-level through the aggregation step. After filtering, the unique `PermitNumber` values within each `IssueYear` are counted. This means a `PermitNumber` that appears in the filtered data more than once (for example, due to amendment records) is counted only once per year.

## 11. Residential Filter

**Rule:** Retain rows where `PropertyUse` contains the string `'Dwelling Uses'`, case-insensitive.

Implementation details:

- **`case=False`** — matches `'Dwelling Uses'`, `'dwelling uses'`, `'DWELLING USES'`, and any mixed-case variant.
- **`na=False`** — rows where `PropertyUse` is null return `False` (excluded) rather than `NaN`. This prevents null rows from slipping through.
- **Substring match** — the rule captures both pure `'Dwelling Uses'` rows and any mixed-use values where `'Dwelling Uses'` appears as part of a longer string.

## 12. Date Filter

**Rule:** Retain rows where `IssueYear` is between 2019 and 2024, inclusive.

- `IssueYear` is a pre-existing integer column in the source file — no date parsing is needed.
- `pandas.Series.between(2019, 2024)` includes both endpoints.
- Rows with a null `IssueYear` return `False` from `.between()` and are excluded automatically.

### Load the Full Dataset — Selected Columns Only

We load the complete permits file (all rows, no `nrows` limit) but restrict to only the three columns needed for this metric: `PermitNumber`, `IssueYear`, and `PropertyUse`. Loading only these columns is faster and uses significantly less memory than reading all 20 columns from a large file.

The `usecols` argument tells pandas to skip all other columns at read time. This cell depends on `PERMITS_PATH` defined in the setup cell above.

In [16]:
df_raw = pd.read_csv(PERMITS_PATH,sep=';',encoding='utf-8-sig',low_memory=False,usecols=['PermitNumber', 'IssueYear', 'PropertyUse'],)

print(f'Full dataset loaded : {len(df_raw):,} rows, {len(df_raw.columns)} columns')
print(f'Columns             : {list(df_raw.columns)}')

Full dataset loaded : 50,995 rows, 3 columns
Columns             : ['PermitNumber', 'PropertyUse', 'IssueYear']


### Apply Residential and Date Filters

Both filters are built as boolean masks and applied together in a single step. We use `.copy()` on the result so the filtered DataFrame is a standalone object — this avoids `SettingWithCopyWarning` if we add columns to it later.

In [ ]:
# Residential filter — substring match, case-insensitive, null-safe
residential_mask = df_raw['PropertyUse'].str.contains('Dwelling Uses', case=False, na=False)

# Date filter — 2019 through 2024 inclusive; returns False for null IssueYear
year_mask = df_raw['IssueYear'].between(2019, 2024)

df_filtered = df_raw[residential_mask & year_mask].copy()

print(f'Rows in full dataset                : {len(df_raw):,}')
print(f'Rows passing residential filter     : {residential_mask.sum():,}')
print(f'Rows passing date filter            : {year_mask.sum():,}')
print(f'Rows after both filters applied     : {len(df_filtered):,}')

Rows in full dataset                : 50,995
Rows passing residential filter     : 36,201
Rows passing date filter            : 30,514
Rows after both filters applied     : 21,944


## 13. Duplicate-Permit Validation

A `PermitNumber` can appear more than once in the source file. Common causes include:

- **Permit amendments** — an existing permit is updated, creating an additional record under the same identifier.
- **Reissuance** — a permit is cancelled and reissued under the same number.
- **Data pipeline artefacts** — duplicate rows introduced during export or loading.

If we aggregate by row count alone, duplicate permit numbers inflate the metric. The official metric therefore uses `nunique()` — counting distinct `PermitNumber` values per year rather than total rows.

Before creating the metric, we quantify the duplicate situation in the filtered data.

### Duplicate-Permit Report

We report five figures:

1. **Total rows** — the number of rows in the filtered DataFrame.
2. **Missing PermitNumber** — rows where `PermitNumber` is null (cannot be counted as a unique permit).
3. **Unique PermitNumber values** — distinct non-null permit identifiers.
4. **Duplicated PermitNumber rows** — rows whose `PermitNumber` appears more than once (counted using `duplicated(keep=False)`, which flags every occurrence, not just the repeats).
5. **Difference** — total rows minus unique count. This equals the number of excess rows above what we would have if every `PermitNumber` appeared exactly once.

In [18]:
total_rows      = len(df_filtered)
missing_pnum    = df_filtered['PermitNumber'].isna().sum()
unique_pnum     = df_filtered['PermitNumber'].nunique()
duplicated_rows = df_filtered['PermitNumber'].duplicated(keep=False).sum()
row_vs_unique   = total_rows - unique_pnum

print(f'Total rows after filtering        : {total_rows:,}')
print(f'Missing PermitNumber              : {missing_pnum:,}')
print(f'Unique PermitNumber values        : {unique_pnum:,}')
print(f'Duplicated PermitNumber rows      : {duplicated_rows:,}')
print(f'Difference (rows - unique count)  : {row_vs_unique:,}')

Total rows after filtering        : 21,944
Missing PermitNumber              : 0
Unique PermitNumber values        : 21,944
Duplicated PermitNumber rows      : 0
Difference (rows - unique count)  : 0


## 14. Create `permit_count_by_year`

We aggregate using `.groupby('IssueYear')['PermitNumber'].nunique()`. This counts the number of distinct `PermitNumber` values within each year — the same permit appearing twice in a year is counted once.

A separate QA series (`row_count`) is calculated using `.size()` — total rows per year. Comparing the two columns shows where (and by how much) the row count exceeds the unique permit count. **`row_count` is not the official metric** — it is included only to make the de-duplication effect visible.

In [19]:
YEAR_RANGE_START = 2019
YEAR_RANGE_END   = 2024

# Official metric: unique PermitNumber per year
permit_count_by_year = (df_filtered
    .groupby('IssueYear')['PermitNumber']
    .nunique()
    .reset_index()
    .rename(columns={'PermitNumber': 'permit_count'})
    .sort_values('IssueYear')
    .reset_index(drop=True))

# QA comparison: row count per year — not the official metric
qa_row_count = (df_filtered
    .groupby('IssueYear')
    .size()
    .reset_index(name='row_count'))

# Merge for side-by-side display
comparison = permit_count_by_year.merge(qa_row_count, on='IssueYear', how='left')
comparison['diff_rows_vs_unique'] = comparison['row_count'] - comparison['permit_count']

print('Official metric : permit_count       (unique PermitNumber per year)')
print('QA column       : row_count          (total rows per year — not the official metric)')
print('diff column     : diff_rows_vs_unique (0 means no duplicate PermitNumbers in that year)')
print()
display(comparison)

Official metric : permit_count       (unique PermitNumber per year)
QA column       : row_count          (total rows per year — not the official metric)
diff column     : diff_rows_vs_unique (0 means no duplicate PermitNumbers in that year)



,IssueYear,permit_count,row_count,diff_rows_vs_unique
0,2019,3731,3731,0
1,2020,3165,3165,0
2,2021,3702,3702,0
3,2022,4501,4501,0
4,2023,3408,3408,0
5,2024,3437,3437,0


## 15. Validation Checks

Six checks confirm the metric is structurally correct:

1. **All six years present** — the output must include every year from 2019 to 2024.
2. **Sorted ascending** — `IssueYear` must increase monotonically.
3. **Positive counts** — every year must have at least one permit.
4. **Sum consistency** — the sum of annual unique counts is reconciled against the total unique permits in the filtered data, accounting for any permit that appears in more than one year.
5. **No missing years** — the year range 2019–2024 must have no gaps.
6. **No null IssueYear** — the filtered analytical population must contain no null year values.

### Check 1 — All Six Years Present and Sorted Ascending

We compare the actual years in `permit_count_by_year` against the expected list `[2019, 2020, 2021, 2022, 2023, 2024]`. A missing year would indicate that no residential permits passed the filters in that year — unlikely but worth catching. We also confirm that the years appear in ascending order.

In [20]:
EXPECTED_YEARS = list(range(YEAR_RANGE_START, YEAR_RANGE_END + 1))
actual_years   = permit_count_by_year['IssueYear'].tolist()

all_present = set(EXPECTED_YEARS) == set(actual_years)
is_sorted   = actual_years == sorted(actual_years)

print(f'Expected years     : {EXPECTED_YEARS}')
print(f'Actual years       : {actual_years}')
print(f'All 6 years present: {all_present}')
print(f'Sorted ascending   : {is_sorted}')

Expected years     : [2019, 2020, 2021, 2022, 2023, 2024]
Actual years       : [2019, 2020, 2021, 2022, 2023, 2024]
All 6 years present: True
Sorted ascending   : True


### Check 2 — Permit Counts Are Positive

Every year in the output must have a permit count greater than zero. A count of zero would mean no unique residential permits were issued in that year after filtering, which is implausible for Vancouver and would indicate a filter or data problem.

In [21]:
min_count    = permit_count_by_year['permit_count'].min()
all_positive = (permit_count_by_year['permit_count'] > 0).all()

print(f'Minimum annual permit count : {min_count:,}')
print(f'All counts positive         : {all_positive}')

if not all_positive:
    print()
    print('WARNING — years with zero or negative count:')
    display(permit_count_by_year[permit_count_by_year['permit_count'] <= 0])

Minimum annual permit count : 3,165
All counts positive         : True


### Check 3 — Sum Consistency

The sum of annual unique permit counts should be close to — but may exceed — the total number of unique `PermitNumber` values in the filtered data. The difference arises when a single `PermitNumber` has records across more than one `IssueYear`: it is counted once per year it appears in, so the annual sums add it up more than once.

A positive `cross_year_count` is informative, not an error. A negative value would indicate a logic problem.

In [22]:
total_unique_permits = df_filtered['PermitNumber'].nunique()
sum_annual_counts    = permit_count_by_year['permit_count'].sum()
cross_year_count     = sum_annual_counts - total_unique_permits

print(f'Unique PermitNumbers in filtered data : {total_unique_permits:,}')
print(f'Sum of annual unique counts           : {sum_annual_counts:,}')
print(f'Apparent cross-year permits           : {cross_year_count:,}')
print()
if cross_year_count == 0:
    print('Sum matches unique total — no permit appears in more than one year.')
elif cross_year_count > 0:
    print(f'{cross_year_count:,} permit(s) appear in more than one IssueYear.')
    print('Annual unique counts are still accurate within each year.')
else:
    print('WARNING: sum is less than unique total — investigate before proceeding.')

Unique PermitNumbers in filtered data : 21,944
Sum of annual unique counts           : 21,944
Apparent cross-year permits           : 0

Sum matches unique total — no permit appears in more than one year.


### Checks 4 and 5 — No Missing Years, No Null IssueYear

**Check 4** confirms there are no gaps in the year sequence 2019–2024 in the output DataFrame. This overlaps with Check 1 but frames the result differently: instead of asking whether the expected years are present, it asks whether any expected year is absent.

**Check 5** confirms that the filtered analytical population contains no null `IssueYear` values. Since we applied a `.between(2019, 2024)` filter — which returns `False` for null — this should always be zero, but we verify explicitly.

In [23]:
# Check 4: no missing years in the output
expected_year_set = set(range(YEAR_RANGE_START, YEAR_RANGE_END + 1))
actual_year_set   = set(permit_count_by_year['IssueYear'])
missing_years     = expected_year_set - actual_year_set

print(f'Check 4 — Missing years in output : {sorted(missing_years) if missing_years else "none"}')

# Check 5: no null IssueYear in the filtered data
null_year_count = df_filtered['IssueYear'].isna().sum()
print(f'Check 5 — Null IssueYear in filtered data : {null_year_count:,}')

if not missing_years and null_year_count == 0:
    print()
    print('Both checks passed.')

Check 4 — Missing years in output : none
Check 5 — Null IssueYear in filtered data : 0

Both checks passed.


## 16. Initial Result

`permit_count_by_year` has been created and validated. The six checks above confirm:

- All six years (2019–2024) are present in the output and sorted in ascending order.
- Every year has a positive permit count.
- The sum of annual unique counts is reconciled against the total unique-permit population in the filtered data.
- No years are missing from the output range.
- No null `IssueYear` values exist in the filtered analytical population.

The metric is ready to be used in the cross-dataset analysis phase.

| Column | Description |
|---|---|
| `IssueYear` | Calendar year the permits were issued (2019–2024) |
| `permit_count` | Count of unique residential `PermitNumber` values issued in that year |

> **Reminder:** `permit_count` measures issued permit identifiers, not completed housing units. A single permit may represent one dwelling or many. This metric is a housing-supply activity signal, not a direct count of new homes added to the city's stock.

The next step is to derive `permit_project_value_by_year` — the sum of declared `ProjectValue` for residential permits per year.

---

## `permit_project_value_by_year` — Annual Residential Permit Project Value

With `permit_count_by_year` validated, we now derive the companion metric: the declared project value associated with residential permits. This adds a monetary dimension to the permit-count signal.

## 17. Business Definition — `permit_project_value_by_year`

**`permit_project_value_by_year`** summarises the declared construction value of residential building permits issued in Vancouver, grouped by year from 2019 to 2024.

Four framing points apply to this metric:

- **`ProjectValue` is declared construction value, not property market value.** It represents the value of the permitted work (labour and materials) as declared by the applicant. It is not the assessed value, sale price, or market value of the property.

- **Totals are in nominal dollars and are not adjusted for inflation.** A dollar of declared construction value in 2019 is not directly comparable to a dollar in 2024. Year-over-year comparisons should account for this limitation.

- **Large projects can skew annual totals.** A single large apartment building can add tens of millions of dollars to one year's total while contributing only one permit to the count. The median per-permit value is included as a robustness metric that is less sensitive to these outliers.

- **This is a housing-supply investment and activity signal, not a count of completed housing units.** Declared value reflects the scale of permitted construction activity, not the number of dwellings delivered.

## 18. Analytical Grain

| Level | Grain | Description |
|---|---|---|
| Source data | One row = one permit record | Each row in the raw file is one permit |
| Output metric | One row = one calendar year | Aggregated to year level |

Within each year the aggregation produces four summary values:

| Column | Aggregation | Note |
|---|---|---|
| `total_project_value` | Sum of `ProjectValue` | Null rows contribute 0 (pandas default) |
| `median_project_value` | Median of `ProjectValue` | Robust to large outliers |
| `mean_project_value` | Mean of `ProjectValue` | Pulled upward by high-value projects |
| `permit_count` | Unique `PermitNumber` count | Consistent with `permit_count_by_year` |

## 19. Residential Filter

**Rule:** Retain rows where `PropertyUse` contains `'Dwelling Uses'`, case-insensitive, with null rows excluded (`na=False`).

This is identical to the rule used for `permit_count_by_year`. See section 11 for the full rationale.

## 20. Date Filter

**Rule:** Retain rows where `IssueYear` is between 2019 and 2024, inclusive. Null `IssueYear` values return `False` from `.between()` and are excluded automatically.

This is identical to the rule used for `permit_count_by_year`. See section 12 for the full rationale.

## 21. `ProjectValue` Meaning

`ProjectValue` is the declared monetary value of the permitted work, as submitted by the permit applicant at the time of application. It is the estimated cost of the construction activity — labour, materials, and related work — not the value of the property itself.

Key characteristics of this field:

- **Self-reported** — applicants declare the value; it is not independently verified at the permit stage.
- **Pre-construction estimate** — the declared value reflects the anticipated cost at application time. Actual costs may differ.
- **Nominal dollars** — values are not adjusted for construction cost inflation across years.
- **Null-possible** — some permits have no declared value, either because the field was optional, the work type does not require a value estimate, or a data entry was omitted.
- **Zero-possible** — a declared value of exactly zero may indicate a nominal fee submission, a data entry convention, or a genuinely no-cost permitted activity.

## 22. Missing-Value Rule

**Rule for `total_project_value`:** We use pandas default `.sum(skipna=True)`, which excludes null `ProjectValue` rows from the sum. A null row is treated as contributing zero to the annual total rather than making the entire year's total null.

This means:

| `ProjectValue` state | Treatment |
|---|---|
| Non-null, positive | Included in sum, median, and mean |
| Zero | Included (contributes 0 to sum; pulled into median and mean) |
| Null | Excluded from sum, median, and mean by pandas default |
| Negative | Included but flagged — reduces the total if present |

The median and mean also use pandas default `skipna=True`, so null rows are excluded from those calculations as well.

Before creating the metric, we report the count of null, zero, and negative `ProjectValue` rows so the quality of the input is transparent.

### Load the Full Dataset — Four Columns

We reload the complete permits file with `usecols` limited to the four columns needed for this metric. This is a fresh load — separate from the three-column load used for `permit_count_by_year` — because we now also need `ProjectValue`.

This cell depends on `PERMITS_PATH` defined in the setup cell above.

In [24]:
df_pv_raw = pd.read_csv(PERMITS_PATH,
    sep=';',
    encoding='utf-8-sig',
    low_memory=False,
    usecols=['PermitNumber', 'IssueYear', 'PropertyUse', 'ProjectValue'],)

print(f'Full dataset loaded : {len(df_pv_raw):,} rows, {len(df_pv_raw.columns)} columns')
print(f'Columns             : {list(df_pv_raw.columns)}')

Full dataset loaded : 50,995 rows, 4 columns
Columns             : ['PermitNumber', 'ProjectValue', 'PropertyUse', 'IssueYear']


### Apply Residential and Date Filters

The same two masks used in section 11–12 are applied here to the new DataFrame. We name the filtered result `df_pv_filtered` to keep it separate from `df_filtered` (the three-column filtered frame used for `permit_count_by_year`).

In [25]:
residential_mask_pv = df_pv_raw['PropertyUse'].str.contains('Dwelling Uses', case=False, na=False)
year_mask_pv = df_pv_raw['IssueYear'].between(2019, 2024)

df_pv_filtered = df_pv_raw[residential_mask_pv & year_mask_pv].copy()

print(f'Rows in full dataset                : {len(df_pv_raw):,}')
print(f'Rows passing residential filter     : {residential_mask_pv.sum():,}')
print(f'Rows passing date filter            : {year_mask_pv.sum():,}')
print(f'Rows after both filters applied     : {len(df_pv_filtered):,}')

Rows in full dataset                : 50,995
Rows passing residential filter     : 36,201
Rows passing date filter            : 30,514
Rows after both filters applied     : 21,944


### `ProjectValue` Quality Report

Before aggregating, we inspect the quality of `ProjectValue` in the filtered data. A **valid** `ProjectValue` is defined here as a non-null, positive number — one that meaningfully contributes to the annual sum. Zeros and negatives are reported separately so their scale is visible before the metric is created.

In [26]:
n_rows          = len(df_pv_filtered)
missing_pv      = df_pv_filtered['ProjectValue'].isna().sum()
missing_pv_pct  = missing_pv / n_rows * 100
zero_pv         = (df_pv_filtered['ProjectValue'] == 0).sum()
zero_pv_pct     = zero_pv / n_rows * 100
negative_pv     = (df_pv_filtered['ProjectValue'] < 0).sum()
valid_pv        = (df_pv_filtered['ProjectValue'] > 0).sum()

print(f'Total rows after filtering              : {n_rows:,}')
print(f'Missing ProjectValue                    : {missing_pv:,}  ({missing_pv_pct:.1f}%)')
print(f'ProjectValue equal to zero              : {zero_pv:,}  ({zero_pv_pct:.1f}%)')
print(f'ProjectValue negative                   : {negative_pv:,}')
print(f'Valid ProjectValue (non-null, positive)  : {valid_pv:,}')

Total rows after filtering              : 21,944
Missing ProjectValue                    : 0  (0.0%)
ProjectValue equal to zero              : 4,472  (20.4%)
ProjectValue negative                   : 0
Valid ProjectValue (non-null, positive)  : 17,472


## 23. Create `permit_project_value_by_year`

We use `.groupby('IssueYear').agg()` to compute four summaries in a single pass. `named aggregation` syntax (`column_name=('source_col', 'func')`) produces output columns with the exact names we want without a rename step.

- **`total_project_value`** — `.sum()` with default `skipna=True`: null rows contribute 0; negative rows reduce the total if any exist.
- **`median_project_value`** — `.median()` with `skipna=True`: the middle declared value in each year, robust to high-value outliers.
- **`mean_project_value`** — `.mean()` with `skipna=True`: the average declared value per permit in each year; can be pulled upward by large projects.
- **`permit_count`** — `.nunique()` on `PermitNumber`: identical to the de-duplication approach used in `permit_count_by_year`.

In [27]:
permit_project_value_by_year = (
    df_pv_filtered
    .groupby('IssueYear')
    .agg(
        total_project_value  = ('ProjectValue', 'sum'),
        median_project_value = ('ProjectValue', 'median'),
        mean_project_value   = ('ProjectValue', 'mean'),
        permit_count         = ('PermitNumber', 'nunique'),
    )
    .reset_index()
    .sort_values('IssueYear')
    .reset_index(drop=True)
)

print('IssueYear range : 2019 – 2024')
print('Value columns   : nominal CAD, not inflation-adjusted')
print()
display(permit_project_value_by_year)

IssueYear range : 2019 – 2024
Value columns   : nominal CAD, not inflation-adjusted



,IssueYear,total_project_value,median_project_value,mean_project_value,permit_count
0,2019,1.673854e+09,35000.0,4.486341e+05,3731
1,2020,1.608853e+09,30900.0,5.083265e+05,3165
2,2021,1.609825e+09,20000.0,4.348527e+05,3702
3,2022,3.720902e+09,25000.0,8.266834e+05,4501
4,2023,3.945624e+09,30000.0,1.157754e+06,3408
5,2024,2.870462e+09,45000.0,8.351649e+05,3437


## 24. Validation Checks

Six checks confirm the metric is structurally correct:

1. **All six years present and sorted** — output covers 2019–2024 in ascending order.
2. **Annual totals non-negative** — each year's `total_project_value` must be ≥ 0.
3. **`permit_count` matches `permit_count_by_year`** — both metrics filter identically, so the per-year unique permit counts must agree. Checked against the previously validated DataFrame if it is available in the notebook runtime.
4. **No null `IssueYear`** — the filtered population must contain no null year values.
5. **No negative `ProjectValue` rows** — any negative row would silently reduce annual totals and warrants investigation.
6. **`permit_count` is positive for all years** — every year must have at least one permit.

### Check 1 — All Six Years Present and Sorted Ascending

In [28]:
pv_expected_years = list(range(2019, 2025))
pv_actual_years   = permit_project_value_by_year['IssueYear'].tolist()

pv_all_present = set(pv_expected_years) == set(pv_actual_years)
pv_is_sorted   = pv_actual_years == sorted(pv_actual_years)

print(f'Expected years     : {pv_expected_years}')
print(f'Actual years       : {pv_actual_years}')
print(f'All 6 years present: {pv_all_present}')
print(f'Sorted ascending   : {pv_is_sorted}')

Expected years     : [2019, 2020, 2021, 2022, 2023, 2024]
Actual years       : [2019, 2020, 2021, 2022, 2023, 2024]
All 6 years present: True
Sorted ascending   : True


### Check 2 — Annual Total Project Value Is Non-Negative

A negative annual total would mean the sum of declared project values is below zero for that year — which would require negative `ProjectValue` entries to outweigh positive ones. We check both the output column and flag the underlying rows.

In [ ]:
neg_total_years = permit_project_value_by_year[permit_project_value_by_year['total_project_value'] < 0]
all_totals_nonneg = len(neg_total_years) == 0

print(f'Years with negative total_project_value : {len(neg_total_years)}')
print(f'All annual totals non-negative           : {all_totals_nonneg}')

if not all_totals_nonneg:
    print()
    print('WARNING — affected years:')
    display(neg_total_years)

Years with negative total_project_value : 0
All annual totals non-negative           : True


### Check 3 — `permit_count` Matches `permit_count_by_year`

Both metrics apply the same residential filter and date filter, so the unique permit count per year should be identical. We merge the two DataFrames on `IssueYear` and compare column by column. If `permit_count_by_year` was not created in this notebook session, we skip the check and print a clear message.

In [30]:
try:
    pv_count_check = permit_count_by_year[['IssueYear', 'permit_count']].merge(permit_project_value_by_year[['IssueYear', 'permit_count']].rename(
            columns={'permit_count': 'pv_permit_count'}),on='IssueYear',how='outer',)
    pv_count_check['counts_match'] = (pv_count_check['permit_count'] == pv_count_check['pv_permit_count'])
    all_counts_match = pv_count_check['counts_match'].all()
    print('permit_count_by_year  vs  permit_project_value_by_year permit_count:')
    display(pv_count_check)
    print(f'\nAll counts match: {all_counts_match}')
    if not all_counts_match:
        print('WARNING: mismatch found — the two sections applied different filters.')
except NameError:
    print('permit_count_by_year not found in notebook runtime.')
    print('Run the permit_count_by_year section first to enable this cross-check.')

permit_count_by_year  vs  permit_project_value_by_year permit_count:


,IssueYear,permit_count,pv_permit_count,counts_match
0,2019,3731,3731,True
1,2020,3165,3165,True
2,2021,3702,3702,True
3,2022,4501,4501,True
4,2023,3408,3408,True
5,2024,3437,3437,True



All counts match: True


### Checks 4, 5, and 6 — Null IssueYear, Negative ProjectValue, Positive Counts

**Check 4** verifies no null `IssueYear` exists in the filtered data (the `.between()` filter should guarantee this, but we confirm explicitly).

**Check 5** counts negative `ProjectValue` rows in the filtered data. Any found here reduce the annual totals and should be investigated — they are not removed at this stage.

**Check 6** confirms that every year in the output has at least one permit, so no year has a zero or null `permit_count`.

In [31]:
# Check 4: no null IssueYear in filtered data
null_year_pv = df_pv_filtered['IssueYear'].isna().sum()
print(f'Check 4 — Null IssueYear in filtered data       : {null_year_pv:,}')

# Check 5: no negative ProjectValue in filtered data
neg_pv_rows = (df_pv_filtered['ProjectValue'] < 0).sum()
print(f'Check 5 — Negative ProjectValue rows             : {neg_pv_rows:,}')
if neg_pv_rows > 0:
    print('          (these reduce annual totals — review before using metric)')

# Check 6: positive permit_count for all years
min_permit_count_pv = permit_project_value_by_year['permit_count'].min()
all_counts_pos_pv   = (permit_project_value_by_year['permit_count'] > 0).all()
print(f'Check 6 — Minimum annual permit_count            : {min_permit_count_pv:,}')
print(f'          All permit counts positive              : {all_counts_pos_pv}')

if null_year_pv == 0 and neg_pv_rows == 0 and all_counts_pos_pv:
    print()
    print('All three checks passed.')

Check 4 — Null IssueYear in filtered data       : 0
Check 5 — Negative ProjectValue rows             : 0
Check 6 — Minimum annual permit_count            : 3,165
          All permit counts positive              : True

All three checks passed.


## 25. Initial Result

`permit_project_value_by_year` has been created and validated. The six checks above confirm:

- All six years (2019–2024) are present in the output and sorted ascending.
- Every year's `total_project_value` is non-negative.
- The per-year `permit_count` matches `permit_count_by_year` — both metrics applied identical filters.
- No null `IssueYear` values in the filtered analytical population.
- Negative `ProjectValue` presence confirmed (or absent) and flagged for review.
- Every year has a positive permit count.

The metric is ready for use in the cross-dataset analysis phase.

| Column | Description |
|---|---|
| `IssueYear` | Calendar year permits were issued (2019–2024) |
| `total_project_value` | Sum of declared `ProjectValue` for residential permits — nominal CAD |
| `median_project_value` | Median declared value per permit — less sensitive to large outliers |
| `mean_project_value` | Mean declared value per permit — pulled upward by high-value projects |
| `permit_count` | Unique residential permit identifiers issued in that year |

> **Reminder:** `total_project_value` is declared construction value, not property market value. Values are in nominal dollars and are not adjusted for inflation. This metric is a housing-supply investment and activity signal — not a count of completed homes.